In [1]:
import os
os.getcwd()

from pathlib import Path
from stgae.config.load_config import load_config
from stgae.data.preproccesing import get_columns
import pandas as pd

paths = load_config()['paths']    
data_path = Path(paths['raw_data'])

Drop the rows where there are missing values. We are interested in predicting full observations, that is in every epoch, all feature for the nodes that are not measured on that epoch, but not specific missing values for current rows. 

In [2]:
df = pd.read_csv(data_path / 'data.txt', names=get_columns(), sep=' ')
df = df.dropna()

Now we will focus on the day where there is less missing data. We can get this day by counting the number of observations per day.

In [3]:
#count rows per day and get max
day = df.groupby('date').size().idxmax()

#filter only that day
filtered = df[df['date'] == day]
filtered.head()

,date,time,epoch,moteid,temperature,humidity,light,voltage
16601,2004-03-08,00:00:34.232157,25805,1.0,22.6932,39.3823,46.92,2.62796
16602,2004-03-08,00:01:02.512187,25806,1.0,22.6834,39.4162,46.92,2.62796
16603,2004-03-08,00:01:42.9882,25807,1.0,22.6834,39.4502,46.92,2.62796
16604,2004-03-08,00:02:08.119067,25808,1.0,22.6638,39.4842,46.92,2.62796
16605,2004-03-08,00:02:43.470276,25809,1.0,22.6638,39.4162,46.92,2.61639


Now I count the average number of mote ids per epoch

In [4]:
filtered.groupby('epoch').size().mean()

np.float64(34.439930555555556)

We have approximately 34 measurements per epoch. Given that the original data comes from 54 sensors... I study if there are particular sensors that have very little data.

In [5]:
number_of_epochs = filtered['epoch'].nunique()
filtered.groupby('moteid').size().sort_values() / number_of_epochs

moteid
12.0    0.352778
30.0    0.377778
14.0    0.448958
53.0    0.458333
32.0    0.496875
39.0    0.501042
50.0    0.504167
8.0     0.530903
52.0    0.533681
19.0    0.534722
33.0    0.542014
27.0    0.553819
13.0    0.557292
6.0     0.576042
16.0    0.579514
54.0    0.581250
41.0    0.583333
2.0     0.607292
43.0    0.629167
34.0    0.632292
25.0    0.632639
42.0    0.646181
49.0    0.649306
40.0    0.665625
4.0     0.668750
37.0    0.672917
17.0    0.697569
51.0    0.698264
35.0    0.704167
29.0    0.712500
10.0    0.735417
26.0    0.744444
1.0     0.763194
31.0    0.764236
24.0    0.773958
9.0     0.777778
3.0     0.780903
38.0    0.783333
11.0    0.786458
44.0    0.789236
18.0    0.807986
23.0    0.810069
36.0    0.820833
46.0    0.852431
45.0    0.858333
21.0    0.863542
7.0     0.864583
20.0    0.866319
48.0    0.873611
22.0    0.894792
47.0    0.899306
dtype: float64

I discarded sensors 12, 30, 14 and 53, given that they have data for less than half of the epochs. To make this decision I also looked at the actual distribution of the sensors, to check that this sensors were not a cluster and all together.

In [6]:
filtered = filtered[(filtered['moteid'] != 12) & (filtered['moteid'] != 30) & (filtered['moteid'] != 14) & (filtered['moteid'] != 53)]
filtered['moteid'].unique()


array([ 1.,  2.,  3.,  4.,  6.,  7.,  8.,  9., 10., 11., 13., 16., 17.,
       18., 19., 20., 21., 22., 23., 24., 25., 26., 27., 29., 31., 32.,
       33., 34., 35., 36., 37., 38., 39., 40., 41., 42., 43., 44., 45.,
       46., 47., 48., 49., 50., 51., 52., 54.])

I observed that there isn't data corresponding to sensors 5 and 28 either.  I will exclude those sensors as well.

Finally, I reindex the epochs to range from 0 to number of epochs - 1 and sensors from 0 to number of sensors - 1. This will be useful for later defining the tensors.

In [7]:
first_epoch = filtered['epoch'].min()
filtered['epoch'] = filtered['epoch'] - first_epoch

assert filtered['epoch'].max() == filtered['epoch'].nunique()-1

In [8]:
#rename moteid
moteid_mapping = {old_id: new_id for new_id, old_id in enumerate(sorted(filtered['moteid'].unique()))}
filtered['moteid'] = filtered['moteid'].map(moteid_mapping)
filtered['moteid'].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46])

Let's build the graph.
I decided to use a graph using knn (with k=5), and the edges weighted according to the inverse of the distance. 

Building adjacency matrix

In [ ]:
from stgae.data.preproccesing import calculate_distances, calculate_adjacency_matrix

coordinates = pd.read_csv(paths['data_root'] / 'sensor_coordinates.txt', sep=' ')

dist_matrix = calculate_distances(coordinates) #numpy array

k=3 #for k-nearest neighbors adj matrix
exclude_sensors = [5, 12, 14, 28, 30, 53]

A = calculate_adjacency_matrix(dist_matrix, k=k, exclude=exclude_sensors) #torch tensor

SyntaxError: expected argument value expression (3936591099.py, line 5)

In [12]:
A.shape

torch.Size([48, 48])

In [10]:
print(dist_matrix.shape)

(54, 54)


Now, I built the tensors to then create the Dataset.
I created X and M, of shapes
X: shape (T, N, F)
M: shape (T, N, 1)
where
X[t, n] = features of sensor n at epoch t
M[t, n] = 1 if sensor exists at epoch t, else 0

In [19]:
from stgae.data.preproccesing import build_tensors
X, M = build_tensors(filtered, epochs=filtered['epoch'].unique(), sensors=filtered['moteid'].unique(), feature_cols=['temperature', 'humidity', 'light', 'voltage'])

Now lets build the dataset.  <br>
For a center time t, and a window size W: <br>
past window: [t-W+1, ..., t] <br>
future window: [t+1, ..., t+W] <br>

Masking: masking sensor level per window, that is, for each timestep, choose a subset of known sensors at that timestep and mask them along the whole window 

In [20]:
from stgae.data.dataset import STBGNNDataset

dataset = STBGNNDataset(X, M, window_size=4)

In [21]:
A.shape

torch.Size([48, 48])

Training the model

In [22]:
import torch
from torch.utils.data import DataLoader
from stgae.model.bistgcn import BiSTGCN
from stgae.model.train import train_step

# Hyperparameters
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# dataloader
dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

model = BiSTGCN(in_features=dataset.F, hidden_dim=32, adj=A).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Starting training on {DEVICE}...")

for epoch in range(EPOCHS):
    total_loss = 0.0
    
    for batch_idx, batch in enumerate(dataloader):
        loss = train_step(batch, model, optimizer, DEVICE)
        total_loss += loss

        break
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_loss:.6f}")
    break

Starting training on cpu...


RuntimeError: einsum(): subscript m has size 47 for operand 1 which does not broadcast with previously seen size 48

In [ ]:
#check example
print(filtered.iloc[0])

print(M[0,0]) #0 (sensor 0 not present at epoch 0)

print(X[1, 0, :])  # sensor 1, epoch 0, all features
print(M[1,0]) #1

date                2004-03-08
time           00:00:34.232157
epoch                        1
moteid                       0
temperature            22.6932
humidity               39.3823
light                    46.92
voltage                2.62796
Name: 16601, dtype: object
[0.]
[22.6932  39.3823  46.92     2.62796]
[1.]


In [15]:
filtered[filtered['epoch']==0] #sensor 11 not present at epoch 0


,date,time,epoch,moteid,temperature,humidity,light,voltage
56028,2004-03-08,00:00:06.855096,0,1,22.9284,40.2976,128.80,2.60491
106904,2004-03-08,00:00:09.900107,0,2,22.7520,39.4842,50.60,2.62796
151749,2004-03-08,00:00:07.797873,0,3,22.8598,40.2299,101.20,2.58226
237627,2004-03-08,00:00:04.397595,0,5,22.4776,40.3652,101.20,2.61639
284326,2004-03-08,00:00:02.542436,0,6,22.0758,40.7031,114.08,2.61639
306752,2004-03-08,00:00:11.283829,0,7,22.0464,42.3170,97.52,2.67532
355055,2004-03-08,00:00:03.280596,0,8,21.9484,42.3170,60.72,2.60491
402311,2004-03-08,00:00:16.419836,0,9,21.0272,44.9782,2.30,2.58226
461244,2004-03-08,00:00:17.455264,0,10,20.8508,45.3083,1.84,2.59354
566678,2004-03-08,00:00:08.757966,0,12,22.1346,41.4444,6.90,2.51661


NEXT STEP (CHECK GPT): Now implement thw windowed dataset